In [1]:
from pyscf import gto, fci, mcscf
import numpy as np
from gpu4pyscf import scf


/home/mert/.local/lib/python3.10/site-packages/gpu4pyscf/lib/cutensor.py:154: UserWarning: using cupy as the tensor contraction engine.
  warnings.warn(f'using {contract_engine} as the tensor contraction engine.')


In [2]:
# Molecular Information part
# !!!!! in CI part mo not forget the change electrons 
# !!!!! also orbitals are important  

xyz = '''
N   0.000   0.000   1.119
H   0.000   0.000   2.122
C   0.000   1.122   0.330
C   0.000  -1.122   0.330
C   0.000   0.711  -0.980
C   0.000  -0.711  -0.980
H   0.000   2.107   0.762
H   0.000  -2.107   0.762
H   0.000   1.357  -1.841
H   0.000  -1.357  -1.841'''
molecule = 'C4H5N'
spin = 0
charge =  0


In [4]:
def rhf_calculation(xyz, molecule, spin, charge):
    mol = gto.M(
          atom= xyz,
          basis = 'sto-6g',
          charge = charge,
          verbose = 5 ,
          spin = spin, 
          symmetry = True,
          output  ="./"+molecule+".txt")

    mf = mol.RHF().run()
    mf.analyze() 
    S_ij = mf.get_ovlp()
    C_ij = mf.mo_coeff
    np.savetxt("C_"+molecule+'mo_coeff.txt', C_ij,fmt = '%.8f')
    np.savetxt('S_'+molecule+'_overlap.txt', S_ij,fmt = '%.8f')

    return mf, mol 

In [5]:
def orb_occ_table(orb_energy,orb_occ):
    Homo = None
    Lumo = None
    print ("orbital  energy   occ")

    for i in range (0,(len(orb_occ))):
        print(i ,"   ", orb_energy[i],"  ",orb_occ[i])
              
    for i in range(len(orb_occ) - 1):
        if orb_occ[i] == 2  and orb_occ[i+1] == 0:
            homo = i
            lumo = i + 1
            print(f"Homo_index: {homo}, Lumo_index: {lumo}")
    return homo, lumo


In [6]:
mf, mol  = rhf_calculation( xyz , molecule, spin, charge )
homo , lumo =orb_occ_table(mf.mo_energy, mf.mo_occ)

output file: ./C4H5N.txt
orbital  energy   occ
0     -15.606516158955325    2.0
1     -11.190754530027917    2.0
2     -11.190626979150833    2.0
3     -11.138854800864323    2.0
4     -11.138382885370376    2.0
5     -1.2278043715269766    2.0
6     -0.9803829570386116    2.0
7     -0.9171683735855523    2.0
8     -0.7368808795121444    2.0
9     -0.7090929964362813    2.0
10     -0.6866991154157858    2.0
11     -0.5323951838860335    2.0
12     -0.5246100373953972    2.0
13     -0.5195532992572955    2.0
14     -0.4879512887557038    2.0
15     -0.46474290027836074    2.0
16     -0.2843026394579655    2.0
17     -0.23998228488895407    2.0
18     0.3162680445329914    0.0
19     0.400272932711863    0.0
20     0.5612792461043434    0.0
21     0.6402791070887506    0.0
22     0.6708002328992354    0.0
23     0.6911798885474445    0.0
24     0.7137042597384107    0.0
25     0.7716491008241648    0.0
26     0.7920545436668848    0.0
27     0.9560650050884776    0.0
28     1.03959896494

In [7]:
n_elec= 6
n_orb = 5 # number of orbital can be determined based on the number of pi  orbitals 
casscf = mcscf.CASSCF(mf, n_orb, n_elec)
'''output in order
e_tot
e_cas
Fci vecs 
mo_coeff
mo_energy'''
mo = casscf.sort_mo([14,17,18,19,20])# the numbers are from the index of pi orbitals 
casscf.kernel(mo) 


(np.float64(-208.3202582670997),
 np.float64(-7.1158538279524635),
 FCIvector([[ 9.50666042e-01,  0.00000000e+00,  1.57465854e-02,
              3.74995849e-03,  2.07572079e-03,  0.00000000e+00,
              0.00000000e+00, -5.29067344e-02, -2.31975012e-03,
              0.00000000e+00],
            [ 0.00000000e+00, -1.68846474e-01,  0.00000000e+00,
              0.00000000e+00,  0.00000000e+00,  3.68121036e-02,
              5.49202956e-02,  0.00000000e+00,  0.00000000e+00,
              1.75765524e-03],
            [ 1.57465854e-02,  0.00000000e+00, -6.58146574e-02,
              1.51714897e-02,  8.97188377e-02,  0.00000000e+00,
              0.00000000e+00, -1.94552031e-03,  6.70839408e-03,
              0.00000000e+00],
            [ 3.74995849e-03,  0.00000000e+00,  1.51714897e-02,
             -4.77128068e-02,  5.72399681e-02,  0.00000000e+00,
              0.00000000e+00,  4.95083589e-03,  1.39541125e-03,
              0.00000000e+00],
            [ 2.07572079e-03,  0.00000000

In [8]:
casscf.fcisolver.max_cycle = 3000
casscf.canonicalize(mo_coeff=None, ci=None, eris=None, sort=False, cas_natorb=False, casdm1=None, verbose=3, with_meta_lowdin=False);

filename = f"{molecule}_Ci_coeffs.txt"

with open(filename, 'w', encoding='utf-8') as f:
    
    f.write("det-alpha,     det-beta,    CI coefficients\n")
    
    
    for c, ia, ib in casscf.fcisolver.large_ci(casscf.ci, n_orb, n_elec, tol=.000000001, return_strs=False):
        # Satır formatını oluşturalım
        line = f" {str(ia):<12} {str(ib):<12} {c:.15f}\n"
        
        #write in file 
        f.write(line)
        #show in terminal 
        print(line.strip())

[0 1 2]      [0 1 2]      0.950666042147776
[0 1 2]      [0 2 3]      0.015746585379175
[0 1 2]      [1 2 3]      0.003749958486452
[0 1 2]      [0 1 4]      0.002075720794802
[0 1 2]      [0 3 4]      -0.052906734403504
[0 1 2]      [1 3 4]      -0.002319750118954
[0 1 3]      [0 1 3]      -0.168846474328096
[0 1 3]      [0 2 4]      0.036812103576874
[0 1 3]      [1 2 4]      0.054920295595132
[0 1 3]      [2 3 4]      0.001757655239502
[0 2 3]      [0 1 2]      0.015746585379175
[0 2 3]      [0 2 3]      -0.065814657377869
[0 2 3]      [1 2 3]      0.015171489655441
[0 2 3]      [0 1 4]      0.089718837697348
[0 2 3]      [0 3 4]      -0.001945520306455
[0 2 3]      [1 3 4]      0.006708394077339
[1 2 3]      [0 1 2]      0.003749958486452
[1 2 3]      [0 2 3]      0.015171489655441
[1 2 3]      [1 2 3]      -0.047712806833686
[1 2 3]      [0 1 4]      0.057239968069631
[1 2 3]      [0 3 4]      0.004950835890941
[1 2 3]      [1 3 4]      0.001395411246356
[0 1 4]      [0 1 2]      